# 🔮 ETS & ARIMA Forecasting
**fpppy Chapters 7–9 · Pattern Recognition for the Rest of Us**

> Two complementary forecasting families: ETS captures level, trend, and seasonality with exponential smoothing; ARIMA models autocorrelation structure after differencing. Together they cover most forecasting needs.

### Required reading
📘 [fpppy Ch 7 (ETS)](https://otexts.com/fpppy/expsmooth.html) · [Ch 8 (ARIMA)](https://otexts.com/fpppy/arima.html) · [Ch 9 (Evaluation)](https://otexts.com/fpppy/accuracy.html)

### What you'll learn
- Naive and seasonal naive baselines
- Simple Exponential Smoothing (SES), Holt's linear, Holt-Winters
- ARIMA: differencing, AR, MA — intuition and fitting
- auto_arima — automatic model selection
- Train/test split for time series, MAE/RMSE/MAPE evaluation

### Dataset: Air passengers + Retail sales
### Time: ~70 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import subprocess, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# Clone course repo to get bundled datasets
if not os.path.exists('pattern-recognition-notebooks'):
    print("Downloading course data...")
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', '--quiet',
         'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"Clone failed: {result.stderr[:200]}")
    else:
        print("✓ Course data ready")
else:
    print("✓ Course data already present")

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '../data'  # fallback for local use

print(f"  Data folder: {DATA_DIR}")
print(f"  Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

# Technique-specific imports
from sklearn.metrics import mean_squared_error


In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: AirPassengers

In [ ]:
# Air passengers — numpy built-in reconstruction (exact Box-Jenkins data)
import pandas as pd, numpy as np
data = [112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range('1949-01', periods=144, freq='MS')
passengers = pd.DataFrame({'Passengers': data}, index=idx)
print(f"Air Passengers: {passengers.shape}")
passengers.head()

In [ ]:
# Train/test split — ALWAYS split time series chronologically, never randomly
# Use last 24 months as test set (standard for monthly data)
split_date = '1958-01-01'

train = passengers[passengers.index < split_date]
test  = passengers[passengers.index >= split_date]

print(f"Training set: {train.index[0].strftime('%b %Y')} to {train.index[-1].strftime('%b %Y')}  ({len(train)} months)")
print(f"Test set:     {test.index[0].strftime('%b %Y')}  to {test.index[-1].strftime('%b %Y')}  ({len(test)} months)")
print()
print("Rule: in time series, future data must NEVER be used to train the model.")
print("Shuffling would leak future information into training — always split by date.")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index, train['Passengers'], color='#1e5fa8', lw=2, label=f'Train ({len(train)} months)')
ax.plot(test.index,  test['Passengers'],  color='#e85d20', lw=2, label=f'Test ({len(test)} months)')
ax.axvline(pd.Timestamp(split_date), color='#888', lw=1.5, ls='--', alpha=0.7)
ax.set_title('Air Passengers — Chronological Train/Test Split')
ax.set_ylabel('Monthly Passengers (thousands)')
ax.legend()
plt.tight_layout()
plt.show()


## 📐 Part 1 — Baseline Models

Before any sophisticated model, establish baselines. If your fancy model can't beat a naive baseline, something is wrong.

- **Naive:** forecast = last observed value
- **Seasonal Naive:** forecast = same period last year
- **Mean:** forecast = historical mean

These are your floor — any real model should do better.

In [ ]:
# Implement baseline forecasts
h = len(test)

naive_fc      = pd.Series([train['Passengers'].iloc[-1]] * h, index=test.index)
seasonal_naive = train['Passengers'].iloc[-12:].values  # last 12 months repeated
snaive_fc     = pd.Series(np.tile(seasonal_naive, 2)[:h], index=test.index)
mean_fc       = pd.Series([train['Passengers'].mean()] * h, index=test.index)

def eval_forecast(name, fc, actual):
    mae  = mean_absolute_error(actual, fc)
    rmse = np.sqrt(mean_squared_error(actual, fc))
    mape = np.mean(np.abs((actual - fc) / actual)) * 100
    return {'Model':name, 'MAE':mae, 'RMSE':rmse, 'MAPE%':mape}

results = []
for name, fc in [('Naive', naive_fc), ('Seasonal Naive', snaive_fc), ('Mean', mean_fc)]:
    results.append(eval_forecast(name, fc, test['Passengers']))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual', alpha=0.8)
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
for fc, color, name in [(naive_fc,'#e85d20','Naive'),
                         (snaive_fc,'#1e5fa8','Seasonal Naive'),
                         (mean_fc,'#888','Mean')]:
    ax.plot(fc.index, fc, color=color, lw=1.5, ls='--', label=name)
ax.set_title('Baseline Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

print("\n=== Baseline Performance ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\n📌 Seasonal Naive is surprisingly hard to beat on strongly seasonal data")

## 📈 Part 2 — Exponential Smoothing (ETS)

ETS models assign exponentially decreasing weights to past observations — recent observations matter more.

- **SES (α):** level only — best for flat series
- **Holt (α, β):** level + trend — best for trending series  
- **Holt-Winters (α, β, γ):** level + trend + seasonality — the full model

The 'ETS' name refers to Error-Trend-Seasonality. Each component can be None, Additive, or Multiplicative.

In [ ]:
# Fit Holt-Winters (additive and multiplicative)
hw_add  = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='add', seasonal_periods=12).fit()
hw_mult = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='mul', seasonal_periods=12).fit()

fc_add  = hw_add.forecast(h)
fc_mult = hw_mult.forecast(h)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--')
ax.plot(fc_add.index,  fc_add,  color='#1e5fa8', lw=2, label='Holt-Winters Additive')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=2, label='Holt-Winters Multiplicative')
ax.fill_between(test.index,
                fc_mult * 0.92, fc_mult * 1.08,
                alpha=0.15, color='#e85d20', label='~80% interval (multiplicative)')
ax.set_title('Holt-Winters Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

results += [eval_forecast('HW Additive',       fc_add,  test['Passengers']),
            eval_forecast('HW Multiplicative',  fc_mult, test['Passengers'])]

print("\n=== ETS vs Baseline ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\nSmoothing parameters (multiplicative):")
print(f"  α (level) = {hw_mult.params['smoothing_level']:.3f}")
print(f"  β (trend) = {hw_mult.params['smoothing_trend']:.3f}")
print(f"  γ (seasonal) = {hw_mult.params['smoothing_seasonal']:.3f}")

## 📉 Part 3 — ARIMA: AutoRegressive Integrated Moving Average

**AR(p):** y_t depends on its past p values
```
y_t = φ₁y_{t-1} + φ₂y_{t-2} + ... + φₚy_{t-p} + ε_t
```

**MA(q):** y_t depends on past q forecast errors
```
y_t = ε_t + θ₁ε_{t-1} + ... + θqε_{t-q}
```

**I(d):** differencing d times to achieve stationarity

**ARIMA(p,d,q)** combines all three. **SARIMA(p,d,q)(P,D,Q)m** adds seasonal components.

In [ ]:
# auto_arima — finds optimal (p,d,q)(P,D,Q) automatically
print("Running auto_arima (may take 30-60 seconds)...")
arima_model = auto_arima(
    train['Passengers'],
    seasonal=True,
    m=12,           # monthly seasonality
    stepwise=True,  # faster search
    information_criterion='aic',
    trace=False,
    error_action='ignore',
    suppress_warnings=True
)
print("\nBest ARIMA model found:")
print(arima_model.summary())

In [ ]:
# Forecast with best ARIMA
arima_fc_vals, conf_int = arima_model.predict(n_periods=h, return_conf_int=True)
arima_fc = pd.Series(arima_fc_vals, index=test.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
ax.plot(arima_fc.index, arima_fc, color='#1a7a45', lw=2, label=f'ARIMA{arima_model.order}{arima_model.seasonal_order}')
ax.fill_between(test.index, conf_int[:,0], conf_int[:,1], alpha=0.15, color='#1a7a45', label='95% CI')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=1.5, ls='--', alpha=0.7, label='HW Multiplicative')
ax.set_title('ARIMA vs Holt-Winters — Air Passengers Forecast')
ax.legend()
plt.tight_layout()
plt.show()

results.append(eval_forecast('ARIMA', arima_fc, test['Passengers']))
print("\n=== Final Model Comparison ===")
df_res = pd.DataFrame(results).sort_values('RMSE')
print(df_res.to_string(index=False, float_format='%.2f'))
print(f"\n🏆 Best model by RMSE: {df_res.iloc[0]['Model']}")

In [ ]:
# Residual diagnostics — good model → white noise residuals
arima_model.plot_diagnostics(figsize=(12, 8))
plt.suptitle('ARIMA Residual Diagnostics\n(residuals should look like white noise)', y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 What to look for:")
print("  Standardized residuals: random scatter around zero, no patterns")
print("  Histogram: approximately normal")
print("  Q-Q plot: points on the diagonal")
print("  Correlogram: no significant spikes (residuals uncorrelated)")

In [ ]:
# @title 📝 Quiz — ETS & ARIMA Forecasting
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What do the three parameters of ARIMA(p,d,q) each represent?
# @markdown **Q2:** Why can you NOT randomly shuffle train/test split for time series?
# @markdown **Q3:** When would you choose Holt-Winters over ARIMA?
# @markdown **Q4:** What does the 'd' in ARIMA represent and when is d=1 used?
# @markdown **Q5:** If residuals show significant ACF spikes, what does that indicate?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_="ETS & ARIMA Forecasting"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")
